# 🎹 Piano MP3 → MIDI
**시작 전:** 런타임 → 런타임 유형 변경 → **T4 GPU**  ·  이후 `Ctrl+F9` 한 번으로 끝.

**1단계** — 설치 (약 30~40초)

In [ ]:
# ── 설치 (재시작 없음, 약 30~40초) ───────────────────────
import subprocess, os

has_gpu = subprocess.run('nvidia-smi', shell=True, capture_output=True).returncode == 0
whl     = 'cu121' if has_gpu else 'cpu'

subprocess.run('pip install uv -q', shell=True, capture_output=True)

pkgs = [
    f'torch torchaudio --index-url https://download.pytorch.org/whl/{whl}',
    'librosa==0.9.2',
    'piano_transcription_inference',
    'pretty_midi',
    'noisereduce',
    'soundfile',
]
# 병렬 설치 (의존성 충돌 없는 것들 한꺼번에)
standalone = 'pretty_midi noisereduce soundfile librosa==0.9.2'
subprocess.run(f'uv pip install {standalone} -q', shell=True, capture_output=True)
subprocess.run(f'uv pip install torch torchaudio '
               f'--index-url https://download.pytorch.org/whl/{whl} -q',
               shell=True, capture_output=True)
subprocess.run('uv pip install piano_transcription_inference -q',
               shell=True, capture_output=True)

print('✅ 설치 완료  |  디바이스:', 'GPU ⚡' if has_gpu else 'CPU 🐢')

**2단계** — MP3 선택 → 변환 → 자동 다운로드  *(다른 곡은 이 셀만 재실행)*

In [ ]:
# ── MP3 업로드 → MIDI 변환 → 다운로드 ───────────────────
import os, glob, subprocess
import numpy as np, librosa, soundfile as sf, noisereduce as nr
import pretty_midi, torch
from google.colab import files
from piano_transcription_inference import PianoTranscription, sample_rate, load_audio

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# 이전 파일 정리
for f in glob.glob('/content/*.mid') + ['/content/audio.wav', '/content/proc.wav']:
    try: os.remove(f)
    except: pass

# 업로드
print('📂 MP3 파일을 선택하세요...')
uploaded = files.upload()
src      = '/content/' + list(uploaded.keys())[0]
TITLE    = list(uploaded.keys())[0].rsplit('.', 1)[0]
WAV      = '/content/audio.wav'
subprocess.run(f'ffmpeg -y -i "{src}" "{WAV}" -loglevel quiet', shell=True)

# 전처리
print('🔧 전처리 중...')
y, sr = librosa.load(WAV, sr=44100, mono=True)
y, _  = librosa.effects.trim(y, top_db=35)
y     = nr.reduce_noise(y=y, sr=sr, y_noise=y[:int(sr*0.3)],
                        stationary=False, prop_decrease=0.75)
rms   = np.sqrt(np.mean(y**2))
y     = np.clip(y * np.clip(10**(-18/20)/(rms+1e-9), 0.1, 10.0), -1.0, 1.0)
PROC  = '/content/proc.wav'
sf.write(PROC, y, 44100)

# AI 전사
print(f'🎹 변환 중... ({DEVICE.upper()})')
audio, _ = load_audio(PROC, sr=sample_rate, mono=True)
transcriptor = PianoTranscription(device=DEVICE)
transcriptor.transcribe(audio, '/content/raw.mid')

# 후처리
midi_raw   = pretty_midi.PrettyMIDI('/content/raw.mid')
midi_clean = pretty_midi.PrettyMIDI(initial_tempo=midi_raw.estimate_tempo())
kept = removed = 0
for inst in midi_raw.instruments:
    ni    = pretty_midi.Instrument(program=inst.program, is_drum=inst.is_drum)
    notes = sorted(inst.notes, key=lambda n: (n.start, n.pitch))
    vt    = max(10, np.percentile([n.velocity for n in notes], 5)) if notes else 10
    seen  = {}
    for n in notes:
        if n.end-n.start < 0.04 or n.velocity < vt \
        or not (21 <= n.pitch <= 108) \
        or (n.pitch in seen and n.start-seen[n.pitch] < 0.05):
            removed += 1; continue
        ni.notes.append(pretty_midi.Note(
            velocity=int(np.clip(n.velocity,1,127)),
            pitch=n.pitch, start=n.start, end=min(n.end, n.start+8.0)))
        seen[n.pitch] = n.start; kept += 1
    ni.control_changes = inst.control_changes
    midi_clean.instruments.append(ni)

safe = ''.join(c if c.isalnum() or c in ' _-' else '_' for c in TITLE)[:50]
OUT  = f'/content/{safe}.mid'
midi_clean.write(OUT)
print(f'✅ {safe}.mid  ({kept:,}개 음표)')

# 다운로드
files.download(OUT)